# impedance EDA

In [1]:
# 도구 불러오기
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from glob import glob
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.font_manager as fm

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [3]:
# 원본 메타데이터 로드
df_meta = pd.read_csv("../../dataset/metadata.csv")

In [4]:
# 전처리 (시간/수치형(Capacity, Re, Rct)/file_num 파생)

def clean_parse_time(x):
    if pd.isna(x):
        return pd.NaT
    
    try:
        s = str(x)
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if len(parts) < 3:
            return pd.NaT
        
        nums = [float(p) for p in parts]
        
        year   = int(nums[0])
        month  = int(nums[1])
        day    = int(nums[2])
        hour   = int(nums[3]) if len(nums) > 3 else 0
        minute = int(nums[4]) if len(nums) > 4 else 0
        
        # 핵심: round 제거
        second = int(nums[5]) if len(nums) > 5 else 0
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hour, minute=minute, second=second)
    
    except Exception as e:
        print(f"[파싱 실패] {x} | {e}")
        return pd.NaT

def clean_numeric(x):
    """수치형 데이터의 대괄호 제거 및 float 변환"""
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [전처리 실행] ---
df = df_meta.copy()

# 1. 시간 데이터 정제 (소수점 초까지 완벽 대응)
df['start_time'] = df['start_time'].apply(clean_parse_time)

# 2. 수치형 데이터 정제 (Capacity, Re, Rct)
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 물리적 순서 정렬 (파일 번호 기준)
df['file_num'] = df['filename'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(['battery_id', 'file_num']).reset_index(drop=True)

# 최종 결과 검증
print(f"전체 데이터 개수: {len(df):,}개")
print(f"시간 결측치(NaT): {df['start_time'].isna().sum()}개")
print(f"수치형 결측치(Cap): {df['Capacity'].isna().sum()}개")


전체 데이터 개수: 7,565개
시간 결측치(NaT): 0개
수치형 결측치(Cap): 4796개


In [5]:
# 'B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data 구성하기
# SOH EOL RUL cycle 파생

# 대상 배터리 ID 리스트
target_battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']

# 데이터 저장할 폴더 경로
data_folder = "../../dataset/data"

# 데이터를 쌓아두기 위한 저장소(Dictionary) 생성
collected_data = {}


### 각 배터리별 EOL 사이클 번호를 먼저 파악 (RUL 계산용)
# SOH 80% 기준을 찾기 위해 메타데이터(df)에서 미리 계산
eol_dict = {}
for b_id in target_battery_ids:
    b_meta = df[(df['battery_id'] == b_id) & (df['type'] == 'discharge')].copy()
    if not b_meta.empty:
        initial_cap = b_meta['Capacity'].iloc[0] # 첫 번째 방전 용량
        # SOH 80% 이하인 첫 번째 행의 인덱스(순번) 찾기
        eol_idx = np.where((b_meta['Capacity'] / initial_cap) * 100 <= 80)[0]
        eol_dict[b_id] = eol_idx[0] + 1 if len(eol_idx) > 0 else np.nan

for battery_id in target_battery_ids:
    # 시간순 정렬 (매칭 오류 방지)
    filtered_df = df[df['battery_id'] == battery_id].sort_values('start_time')
    cycle_counters = {'charge': 1, 'discharge': 1, 'impedance': 1}
    
    # 기준 용량 정의 (해당 배터리의 첫 번째 방전 용량)
    dis_rows = filtered_df[filtered_df['type'] == 'discharge']
    first_cap = dis_rows['Capacity'].iloc[0] if not dis_rows.empty else None

    for _, row in filtered_df.iterrows():
        file_path = os.path.join(data_folder, row['filename'])

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)
            d_type = row['type']
            current_cycle = cycle_counters[d_type]

            # --- [1단계: 파생 변수 로직 정의] ---
            # SOH 정의: 오직 discharge 타입일 때만 해당 사이클의 용량으로 계산
            if d_type == 'discharge' and first_cap and pd.notnull(row['Capacity']):
                soh_val = (row['Capacity'] / first_cap) * 100
            else:
                soh_val = np.nan # 다른 타입은 우선 NaN 처리
            
            # EOL 정의: 미리 계산된 eol_dict 이용
            eol_val = eol_dict.get(battery_id)
            
            # RUL 정의: (사망 사이클 - 현재 사이클)
            rul_val = (eol_val - current_cycle) if pd.notnull(eol_val) else np.nan
            if pd.notnull(rul_val): rul_val = max(0, rul_val)

            # --- [2단계: 데이터프레임 열 추가] ---
            temp_df['start_time'] = row['start_time']
            temp_df['battery_id'] = battery_id
            temp_df['type'] = d_type
            temp_df['ambient_temperature'] = row['ambient_temperature']
            
            temp_df['cycle'] = current_cycle
            temp_df['SOH'] = soh_val  # 정의된 값 주입
            temp_df['EOL_cycle'] = eol_val
            temp_df['RUL'] = rul_val

            # 타입별 특화 데이터 추가
            if d_type == 'discharge':
                temp_df['Capacity'] = row['Capacity']
            elif d_type == 'impedance':
                temp_df['Re'] = row['Re']
                temp_df['Rct'] = row['Rct']

            # 카운터 증가 및 저장
            cycle_counters[d_type] += 1
            key = f"df_{d_type}_{battery_id}"
            if key not in collected_data:
                collected_data[key] = []
            collected_data[key].append(temp_df)

# --- [3단계: 데이터 통합 및 결측치 전파] ---
for key, df_list in collected_data.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # SOH 전파: discharge 파일들 사이의 간극(charge, impedance 등)을 메워줌
    # 단, 사용자님이 'discharge에만' 있길 원하신다면 이 섹션을 제외하면 됩니다.
    # 만약 모든 행에 SOH가 필요하다면 아래 ffill/bfill을 유지하세요.
    if 'SOH' in combined_df.columns:
        combined_df['SOH'] = combined_df['SOH'].ffill()
        combined_df['SOH'] = combined_df['SOH'].bfill()
    
    globals()[key] = combined_df
    print(f"✅ {key} 생성 완료 | 파일 {len(df_list)}개 통합 | 크기: {combined_df.shape} | SOH 결측치: {combined_df['SOH'].isnull().sum()} | cycle/SOH/EOL/RUL 파생 완료")


✅ df_charge_B0005 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0005 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0005 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0006 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0006 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0006 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0007 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0007 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0007 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0018 생성 완료 | 파일 134개 통합 | 크기: (279810, 14) | SOH 결측치: 279810 | cycle/SOH/EOL/RUL 파생 완료
✅ df_i

In [ ]:
# 대표로 하나 지정하여 null값 확인
df_impedance_B0005.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13344 entries, 0 to 13343
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Sense_current        13344 non-null  object        
 1   Battery_current      13344 non-null  object        
 2   Current_ratio        13344 non-null  object        
 3   Battery_impedance    13344 non-null  object        
 4   Rectified_Impedance  10842 non-null  object        
 5   start_time           13344 non-null  datetime64[ns]
 6   battery_id           13344 non-null  object        
 7   type                 13344 non-null  object        
 8   ambient_temperature  13344 non-null  int64         
 9   cycle                13344 non-null  int64         
 10  SOH                  0 non-null      float64       
 11  EOL_cycle            13344 non-null  int64         
 12  RUL                  13344 non-null  int64         
 13  Re                   13344 non-

In [7]:
# 간단 확인
display(df_charge_B0006)
display(df_discharge_B0007)
display(df_impedance_B0005) 

,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL
0,3.864624,0.000082,24.682214,-0.001,-0.007,0.000,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
1,3.469113,-4.059185,24.695407,-4.060,1.558,2.532,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
2,3.994806,1.513750,24.711491,1.506,4.710,5.500,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
3,4.005888,1.511389,24.739672,1.506,4.726,8.344,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
4,4.012944,1.510817,24.753180,1.506,4.737,11.125,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541168,0.979650,-0.005157,22.984433,0.000,-0.007,0.000,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541169,-0.001422,-0.001326,22.984099,-0.001,-0.007,2.547,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541170,4.985089,0.000203,22.986148,0.000,4.996,5.500,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541171,4.985345,0.000329,22.994625,-0.001,4.996,8.312,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL,Capacity
0,4.199360,-0.001866,23.937044,-0.0004,0.000,0.000,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
1,4.199497,-0.002139,23.924074,-0.0004,4.215,16.781,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
2,3.985606,-1.988778,24.004257,-2.0000,3.003,35.703,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
3,3.963247,-1.992558,24.162868,-2.0000,2.987,53.781,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
4,3.946647,-1.988491,24.346368,-2.0000,2.972,71.922,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50280,3.336677,-0.002464,38.744012,0.0006,0.001,2781.312,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50281,3.349952,-0.005358,38.462399,0.0006,0.001,2791.062,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50282,3.362104,-0.003906,38.246805,0.0006,0.001,2800.828,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50283,3.373357,-0.002763,37.970504,0.0006,0.001,2810.640,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455


,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL,Re,Rct
0,(-1+1j),(-1+1j),(1+0j),(-0.43892624830326377-0.107298295835479j),(0.07006937798290404-0.00047998469078178944j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
1,(820.6094970703125-36.23455047607422j),(337.0914611816406-82.9207763671875j),(2.3204145178633437+0.4633045948164565j),(0.13008840651776496-0.19711481029612374j),(0.06817886114940203-0.001190040925296937j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
2,(827.2421875-48.23122787475586j),(330.6315612792969-70.01371765136719j),(2.424192647592199+0.36746495469515333j),(0.058770560504133235+0.03330656583655633j),(0.06793257733714593-5.6826811936507056e-05j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
3,(827.1934814453125-56.195716857910156j),(330.8086242675781-61.73442459106445j),(2.4470021712116985+0.28677775364826635j),(0.0058135116366746726-0.060546548141956195j),(0.06691839226387165-0.0008787264015490232j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
4,(824.9295043945312-53.241477966308594j),(332.68267822265625-57.62901306152344j),(2.434304977711638+0.2616460702282485j),(0.12608106668700975-0.09044390544679616j),(0.06807105294348659-0.0001974802021297548j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13339,(915.489013671875-64.04512023925781j),(230.14950561523438+91.9098892211914j),(3.334834919457856-1.6100379067382284j),(0.2450237439418051-0.04983619374573483j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13340,(916.7255249023438+2.986217498779297j),(212.18885803222656+107.74581146240234j),(3.4403927232922893-1.732898190940107j),(0.26459423551817796-0.055235088103719486j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13341,(914.61962890625+126.11148071289062j),(176.59803771972656+131.6827850341797j),(3.6706560245778346-2.0229597798418966j),(0.28857059933181783-0.058837097064953735j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13342,(880.3408203125+293.8252868652344j),(136.84762573242188+146.8810272216797j),(4.060163697383427-2.2107488242884408j),(0.3176999700278338-0.05912720702491042j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792


# 분석

# impedance 분석

배터리 4개를 모두 합침

In [10]:
# 2. 임피던스(Impedance) 데이터 통합
battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_impedance_dfs = []

for b_id in battery_ids:
    df_name = f"df_impedance_{b_id}"
    if df_name in globals():
        # 복사본 생성 및 ID 추가
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id
        all_impedance_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_impedance = pd.concat(all_impedance_dfs, ignore_index=True)

print(f"Impedance 통합 - 전체 행 개수: {len(df_battery_impedance)}")
print(f"포함된 배터리 id: {df_battery_impedance['battery_id'].unique()}")

Impedance 통합 - 전체 행 개수: 42576
포함된 배터리 id: ['B0005' 'B0006' 'B0007' 'B0018']


# 3개 흐름 분석